In [1]:
from pathlib import Path
import tempfile

from embedding_lab.chunking import Chunk
from embedding_lab.vector_store import ChromaVectorStore


class CountingEmbeddings:
    def __init__(self):
        self.embedded_texts: list[str] = []

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        self.embedded_texts.extend(texts)
        return [
            [1.0, 0.0] if "幕墙" in text else [0.0, 1.0]
            for text in texts
        ]

    def embed_query(self, text: str) -> list[float]:
        return self.embed_documents([text])[0]


def make_chunk(chunk_id: str, text: str, source: str) -> Chunk:
    return Chunk(
        id=chunk_id,
        text=text,
        metadata={
            "namespace": "incremental_test",
            "source": source,
            "heading": source,
            "doc_scope": "generation",
        },
    )

In [2]:
persist_dir = Path(tempfile.mkdtemp(prefix="studyagent-incremental-"))
store = ChromaVectorStore(
    persist_dir=persist_dir,
    collection_name="incremental_demo",
    namespace="incremental_test",
    index_signature="teaching-model-v1-chunk-v1",
)
embeddings = CountingEmbeddings()

chunks_v1 = [
    make_chunk("wall-v1", "幕墙需要父墙体", "walls.md"),
    make_chunk("stair-stable", "楼梯连接两个楼层", "stairs.md"),
]

In [3]:
first = store.sync(chunks_v1, embeddings)
assert first.added == 2
assert first.deleted == 0
assert len(embeddings.embedded_texts) == 2

print("第一次：", first)
print("累计向量化文本：", embeddings.embedded_texts)

第一次： SyncStats(total=2, added=2, deleted=0)
累计向量化文本： ['幕墙需要父墙体', '楼梯连接两个楼层']


In [4]:
second = store.sync(chunks_v1, embeddings)
assert second.added == 0
assert second.deleted == 0
assert len(embeddings.embedded_texts) == 2

print("第二次：", second)
print("累计向量化文本数：", len(embeddings.embedded_texts))

第二次： SyncStats(total=2, added=0, deleted=0)
累计向量化文本数： 2


In [5]:
# 模拟墙体正文发生变化：真实 Chunker 会因为 content_hash 变化生成新 ID。
chunks_v2 = [
    make_chunk("wall-v2", "幕墙必须记录 parentWall", "walls.md"),
    make_chunk("stair-stable", "楼梯连接两个楼层", "stairs.md"),
]

third = store.sync(chunks_v2, embeddings)
assert third.added == 1
assert third.deleted == 1
assert len(embeddings.embedded_texts) == 3
assert store.status()["count"] == 2

print("修改后：", third)
print("真正向量化过的文本：", embeddings.embedded_texts)

修改后： SyncStats(total=2, added=1, deleted=1)
真正向量化过的文本： ['幕墙需要父墙体', '楼梯连接两个楼层', '幕墙必须记录 parentWall']
